In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [90]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [91]:
df = pd.read_csv("/content/drive/MyDrive/myprojectdataset.csv")
print("Initial rows:", len(df))
print(df.head())

Initial rows: 559
   study_hours  attendance  assignments_completed  previous_grade  \
0         12.5          85                      8              78   
1          9.2          70                      6              65   
2          6.0          50                      3              45   
3         15.0          95                     10              90   
4          8.0          60                      5              55   

   participation  pass  
0              6     1  
1              5     1  
2              3     0  
3              9     1  
4              4     0  


In [92]:
# Features & target
features = ['study_hours','attendance','assignments_completed','previous_grade','participation']
X = df[features].values
y = df['pass'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)
features = ["study_hours", "attendance", "assignments_completed", "previous_grade", "participation"]
target = "pass"

X = df[features].values
y = df[target].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

print("Shapes -> X_train:", X_train.shape, "y_train:", y_train.shape)

X_train shape: torch.Size([447, 5]) y_train shape: torch.Size([447, 1])
Shapes -> X_train: torch.Size([447, 5]) y_train: torch.Size([447, 1])


In [93]:
class SimpleANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = SimpleANN(input_dim=X_train.shape[1]).to(device)


In [94]:
criterion = nn.BCELoss()  # Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [95]:
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        with torch.no_grad():
            preds = (outputs > 0.5).float()
            acc = (preds == y_train).sum().item() / y_train.size(0)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}, Train Accuracy: {acc*100:.2f}%")


Epoch 10/100, Loss: 0.6441, Train Accuracy: 46.09%
Epoch 20/100, Loss: 0.5892, Train Accuracy: 68.23%
Epoch 30/100, Loss: 0.5282, Train Accuracy: 79.42%
Epoch 40/100, Loss: 0.4623, Train Accuracy: 95.53%
Epoch 50/100, Loss: 0.3943, Train Accuracy: 98.88%
Epoch 60/100, Loss: 0.3298, Train Accuracy: 99.11%
Epoch 70/100, Loss: 0.2707, Train Accuracy: 99.11%
Epoch 80/100, Loss: 0.2201, Train Accuracy: 99.11%
Epoch 90/100, Loss: 0.1791, Train Accuracy: 99.11%
Epoch 100/100, Loss: 0.1468, Train Accuracy: 99.11%


In [96]:
model.eval()
with torch.no_grad():
    y_pred_prob = model(X_test)
    y_pred = (y_pred_prob > 0.5).float()
    acc = (y_pred == y_test).sum().item() / y_test.size(0)

    print(f"Test Accuracy: {acc*100:.2f}%")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test.cpu(), y_pred.cpu()))


Test Accuracy: 99.11%
Confusion Matrix:
[[40  0]
 [ 1 71]]


In [99]:
# Example new student
# study_hours, attendance, assignments_completed, previous_grade, participation
new_student = np.array([[3, 60, 8, 78, 6]])

# Scale & convert
new_scaled = scaler.transform(new_student)
new_tensor = torch.tensor(new_scaled, dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    prob = model(new_tensor).item()
    pred_class = "Pass" if prob > 0.5 else "Fail"

print(f"Predicted Pass Probability: {prob:.2f}")
print(f"Predicted Class: {pred_class}")


Predicted Pass Probability: 0.49
Predicted Class: Fail


In [100]:
import pickle

# Save model + scaler
with open("/content/pass_fail_model.pkl", "wb") as f:
    pickle.dump({
        "model_state": model.state_dict(),
        "scaler": scaler,
        "input_dim": X_train.shape[1]
    }, f)

print("Model saved as pass_fail_model.pkl")


Model saved as pass_fail_model.pkl


In [101]:
from google.colab import files
files.download("/content/pass_fail_model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>